# Training
This notebook trains all three models on the prepared dataset and generates predictions.

In [ ]:
# importing all libraries required for data preparation and model training
# warnings.filterwarnings('ignore') suppresses non-critical deprecation warnings
# that would clutter the output without affecting the results
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

### Setup — Prepare Data & Models
*(Runs the same steps as data_pipeline.ipynb and model.ipynb so this notebook works on its own.)*

In [ ]:
# loading the dataset and removing non-informative columns
# PatientID and DoctorInCharge carry no signal for predicting Alzheimer's
df = pd.read_csv('../data/alzheimers_disease_data.csv')
df = df.drop(columns=[c for c in ['PatientID', 'DoctorInCharge'] if c in df.columns])

# separating the target variable (y) from the input features (X)
# get_dummies encodes any remaining categorical columns as binary indicator variables
y = df['Diagnosis'].astype(int)
X = pd.get_dummies(df.drop(columns=['Diagnosis']), drop_first=True)

# 80/20 train-test split with stratification to preserve class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# StandardScaler is fitted on training data only
# applying the same transformation to test data prevents information leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Data ready — X_train:', X_train.shape, '| X_test:', X_test.shape)

In [ ]:
# defining the three models that will be trained and compared
# SCALED_MODELS marks which models require standardized input due to their sensitivity to feature scale
SCALED_MODELS = {'Logistic Regression'}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                         eval_metric='logloss', random_state=42, n_jobs=-1),
}

### Train

In [ ]:
# training all three models in a single loop to avoid repeating the same code
# model.fit() learns the parameters from the training data
# Logistic Regression receives scaled features; tree-based models use the raw features
fitted = {}

for name, model in models.items():
    if name in SCALED_MODELS:
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train, y_train)

    fitted[name] = model
    print(f'{name}: training complete')

### Predict

In [ ]:
# generating predictions on the held-out test set for each trained model
# model.predict() returns the predicted class labels (0 or 1) for each test sample
# each model receives the same feature format it was trained on (scaled or unscaled)
predictions = {}

for name, model in fitted.items():
    if name in SCALED_MODELS:
        predictions[name] = model.predict(X_test_scaled)
    else:
        predictions[name] = model.predict(X_test)

    # printing the first 10 predictions as a quick sanity check
    print(f'{name}: {predictions[name][:10]}  ... ({len(predictions[name])} predictions total)')